In [ ]:
# choosing piano note

note_nr = 69 # note A4 (440Hz)


filename = "note_reference_" + str(note_nr)
piano = "Shigeru_Kawai_CG"

def set_filenames():
    global filename
    global piano  
    global originalMIDI
    global originalTXT
    global originalPNG
    global originalAUDIO
    global generatedMIDI
    global generatedTXT
    global generatedPNG
    global generatedAUDIO
    global generatedTempTXT
    global postprocessedMIDI
    global logmelSPECTROGRAM
    global batchmormalSPECTROGRAM
    generatedMIDI = filename + "_" + piano + ".mid"
    generatedPNG = filename + "_" + piano + ".png"
    generatedTXT = filename + "_" + piano + ".txt"
    originalMIDI = filename + ".midi"
    originalTXT = filename + ".txt"
    originalPNG = filename + ".png"
    originalAUDIO = filename + ".wav"
    postprocessedMIDI = ""
    generatedAUDIO = filename + "_out.wav"
    generatedTempTXT = "output.txt"
    logmelSPECTROGRAM = "logmel.csv"
    batchmormalSPECTROGRAM = "batchnormal.csv"
    
    
set_filenames()
print("original MIDI filename:", originalMIDI)
print("original TXT filename:", originalTXT)
print("original PNG filename:", originalPNG)
print("original AUDIO filename:", originalAUDIO)
print("generated MIDI filename:", generatedMIDI)
print("generated TXT filename:", generatedTXT)
print("generated TempTXT filename:", generatedTempTXT)
print("generated PNG filename:", generatedPNG)
from IPython.display import Audio


In [ ]:
# Generating reference MIDI and AUDIO files

import os
import subprocess
import random
from mido import Message, MidiFile, MidiTrack, MetaMessage

def generate_midi(note_nr, random_flag=0):
    """
    Generates a MIDI file (and renders a WAV file using fluidsynth) for a given MIDI note.
    
    Parameters:
      note_nr (int): MIDI note number.
      random_flag (int): If set to 1, velocities are shuffled.
    """
    # Hardcoded timing values (ticks)
    note_duration = 288  # on duration: 300ms
    gap_duration = 192   # off duration: 200ms


    # Create a filename based on the parameters.
    if random_flag == 1:
        filename = "note_reference_random_" + str(note_nr)
    else:
        filename = "note_reference_" + str(note_nr)
    
   
    # Define filenames.
    originalMIDI = filename + ".midi"
    originalAUDIO = filename + ".wav"
    
    # Create a new MIDI file and add two tracks.
    mid = MidiFile(ticks_per_beat=480)
    piano_track = MidiTrack()
    pedal_track = MidiTrack()
    mid.tracks.append(piano_track)
    mid.tracks.append(pedal_track)
    
    # Set a tempo (500000 microseconds per beat corresponds to 120 BPM).
    piano_track.append(MetaMessage('set_tempo', tempo=500000, time=0))
    
    # Generate a list of MIDI velocity values (0 to 127) and shuffle if random_flag is enabled.
    velocities = list(range(128))
    if random_flag == 1:
        random.shuffle(velocities)
    
    # Insert an initial note with velocity 127.
    piano_track.append(Message('note_on', note=note_nr, velocity=127, time=0))
    piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))
    
    # For each velocity, add the note with the gap and note duration.
    for i, velocity in enumerate(velocities):
        piano_track.append(Message('note_on', note=note_nr, velocity=velocity, time=gap_duration))
        piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))
        if i < len(velocities) - 1:
            piano_track.append(Message('note_on', note=note_nr, velocity=127, time=gap_duration))
            piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))
    
    # Save the MIDI file.
    mid.save(originalMIDI)
    print(f"MIDI file '{originalMIDI}' generated successfully!")
    

    # Render the MIDI file to a WAV file using FluidSynth with the specified soundfont.
    fluidsynth_path = r"SOUNDFONTS/bin/fluidsynth.exe"
    soundfont_path = r"SOUNDFONTS/KAWAI_MP11SE_SK_Concert_Grand.sf2"

    fluidsynth_cmd = (
        f'"{fluidsynth_path}" -ni "{soundfont_path}" "{originalMIDI}" '
        f'-F "{originalAUDIO}" -r 44100 -g 1'
    )

    subprocess.run(fluidsynth_cmd, shell=True, check=True)
    print(f"WAV file '{originalAUDIO}' generated successfully!")

generate_midi(note_nr=note_nr, random_flag=0)


In [ ]:
#Plotting MIDI events from the reference, original MIDI file

def midi_to_text_and_plot(midi_path, text_path, plot_path, title_text):
    """
    Decodes a MIDI file, writes its events into a text file, and plots:
    1) Velocity vs. Time (ignoring velocity 0) with Y-axis from 0 to 127
    2) Histogram of Velocity Values with 128 bins and X-axis from 0 to 127
    3) MIDI Note Numbers vs. Time mapped to piano keys (1-88)
    4) Histogram of MIDI Note Numbers (mapped to 1-88) with 88 bins
    Args:
        midi_path (str): Path to the input MIDI file.
        text_path (str): Path to save the output text file.
        plot_path (str): Path to save the generated plot as a PNG file.
        title_text (str): Text to be displayed in the upper left corner of the plot.
    """
    import mido
    import matplotlib.pyplot as plt

    midi_file = mido.MidiFile(midi_path)
    ticks_per_beat = midi_file.ticks_per_beat
    tempo = 500000  # Default 120 BPM tempo (500,000 microseconds per beat)

    # Store extracted data
    velocity_values = []
    velocity_times = []
    note_numbers = []
    note_times = []

    with open(text_path, 'w') as text_file:
        text_file.write(f"MIDI File: {midi_path}\n")
        text_file.write(f"Ticks per Beat: {ticks_per_beat}\n\n")

        absolute_ticks = 0  # Tracks cumulative time in ticks

        for i, track in enumerate(midi_file.tracks):
            text_file.write(f"Track {i}: {track.name}\n")
            text_file.write("-" * 40 + "\n")

            for msg in track:
                absolute_ticks += msg.time  # Convert relative to absolute time

                # Tempo Change Handling (Optional)
                if msg.type == 'set_tempo':
                    tempo = msg.tempo  # Set new tempo

                # Convert MIDI ticks to seconds
                seconds = mido.tick2second(absolute_ticks, ticks_per_beat, tempo)

                # Note-On Event (Velocity > 0)
                if msg.type == 'note_on' and msg.velocity > 0:
                    velocity_values.append(msg.velocity)  # Store velocity
                    velocity_times.append(seconds)         # Store time for velocity
                    note_numbers.append(msg.note)            # Store MIDI note number
                    note_times.append(seconds)               # Store time for note

                    text_file.write(f"Time {seconds:.3f} sec | NOTE_ON  | "
                                    f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n")

                # Note-Off Event
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    text_file.write(f"Time {seconds:.3f} sec | NOTE_OFF | "
                                    f"Note: {msg.note:3d} | Velocity: 0\n")

            text_file.write("\n")

    # Map MIDI note numbers to piano keys: MIDI 21 -> 1, MIDI 108 -> 88.
    mapped_note_numbers = [n - 20 for n in note_numbers]

    # Generate Plots as Subplots if there is any velocity data
    if velocity_values:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Top Left: Velocity vs. Time (set Y-axis from 0 to 127)
        axes[0, 0].scatter(velocity_times, velocity_values, color='b', alpha=0.7, label="Note Velocity")
        axes[0, 0].set_xlabel("Time (seconds)")
        axes[0, 0].set_ylabel("Velocity (0-127)")
        axes[0, 0].set_title("Velocity vs. Time")
        axes[0, 0].legend()
        axes[0, 0].grid()
        axes[0, 0].set_ylim(0, 127)
        axes[0, 0].set_xlim(left=0)
        axes[0, 0].text(-0.07, 1.1, title_text, transform=axes[0, 0].transAxes, fontsize=12, verticalalignment='top')

        # Bottom Left: MIDI Note Numbers vs. Time (mapped to 1-88)
        axes[1, 0].scatter(note_times, mapped_note_numbers, color='purple', alpha=0.7, label="Piano Key")
        axes[1, 0].set_xlabel("Time (seconds)")
        axes[1, 0].set_ylabel("Piano Key (1-88)")
        axes[1, 0].set_title("MIDI Note Numbers vs. Time")
        axes[1, 0].legend()
        axes[1, 0].grid()
        axes[1, 0].set_xlim(left=0)
        axes[1, 0].set_ylim(1, 88)

        # Top Right: Histogram of Velocity Values (set X-axis from 0 to 127)
        axes[0, 1].hist(velocity_values, bins=128, range=(0, 127), color='g', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel("Velocity")
        axes[0, 1].set_ylabel("Number of Occurrences")
        axes[0, 1].set_title("Histogram of Velocity Values")
        axes[0, 1].grid()
        axes[0, 1].set_xlim(0, 127)

        # Bottom Right: Histogram of MIDI Note Numbers (mapped to 1-88)
        axes[1, 1].hist(mapped_note_numbers, bins=88, range=(1, 88), color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel("Piano Key (1-88)")
        axes[1, 1].set_ylabel("Number of Occurrences")
        axes[1, 1].set_title("Histogram of MIDI Note Numbers")
        axes[1, 1].grid()
        axes[1, 1].set_xlim(1, 88)

        plt.tight_layout()
        plt.savefig(plot_path)  # Save plot as PNG file
        plt.show()

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {text_path}")
midi_to_text_and_plot(originalMIDI, originalTXT, originalPNG, filename )

In [ ]:
# Playing out the reference, original AUDIO
print(originalAUDIO)
Audio(originalAUDIO)

In [ ]:

# Creating spectrograms

from pathlib import Path
from typing import Optional

import librosa
import numpy as np
import torch
from torchlibrosa.stft import LogmelFilterBank, Spectrogram


# Matched to bytedance/piano_transcription
SAMPLE_RATE = 16000
WINDOW_SIZE = 2048
FRAMES_PER_SECOND = 100
HOP_SIZE = SAMPLE_RATE // FRAMES_PER_SECOND  # 160
MEL_BINS = 229
FMIN = 30
FMAX = SAMPLE_RATE // 2  # 8000
WINDOW = "hann"
CENTER = True
PAD_MODE = "reflect"
REF = 1.0
AMIN = 1e-10
TOP_DB = None
BN_MOMENTUM = 0.01


def init_bn(bn: torch.nn.BatchNorm2d) -> None:
    bn.bias.data.fill_(0.0)
    bn.weight.data.fill_(1.0)


class PianoTranscriptionFrontendWithBN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.spectrogram_extractor = Spectrogram(
            n_fft=WINDOW_SIZE,
            hop_length=HOP_SIZE,
            win_length=WINDOW_SIZE,
            window=WINDOW,
            center=CENTER,
            pad_mode=PAD_MODE,
            freeze_parameters=True,
        )
        self.logmel_extractor = LogmelFilterBank(
            sr=SAMPLE_RATE,
            n_fft=WINDOW_SIZE,
            n_mels=MEL_BINS,
            fmin=FMIN,
            fmax=FMAX,
            ref=REF,
            amin=AMIN,
            top_db=TOP_DB,
            freeze_parameters=True,
        )
        self.bn0 = torch.nn.BatchNorm2d(MEL_BINS, momentum=BN_MOMENTUM)
        init_bn(self.bn0)

    @torch.no_grad()
    def forward(self, waveform: torch.Tensor):
        """
        Args:
            waveform: shape (batch, samples) or (samples,)

        Returns:
            raw_logmel: shape (batch, time_frames, mel_bins)
            post_bn_logmel: shape (batch, time_frames, mel_bins)
        """
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)

        spec = self.spectrogram_extractor(waveform)   # (batch, 1, time, freq)
        raw = self.logmel_extractor(spec)             # (batch, 1, time, mel)

        x = raw.transpose(1, 3)                       # (batch, mel, time, 1)
        x = self.bn0(x)                               # (batch, mel, time, 1)
        post_bn = x.transpose(1, 3)                   # (batch, 1, time, mel)

        return raw.squeeze(1), post_bn.squeeze(1)


def load_audio_mono(audio_path: Path):
    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    waveform, sr = librosa.load(str(audio_path), sr=None, mono=True)
    if waveform.size == 0:
        raise ValueError(f"Loaded empty audio from: {audio_path}")
    return waveform.astype(np.float32, copy=False), sr


def resample_if_needed(waveform: np.ndarray, orig_sr: int) -> np.ndarray:
    if orig_sr == SAMPLE_RATE:
        return waveform
    return librosa.resample(
        y=waveform,
        orig_sr=orig_sr,
        target_sr=SAMPLE_RATE
    ).astype(np.float32, copy=False)


def slice_audio(
    waveform: np.ndarray,
    sr: int,
    start_sec: float = 0.0,
    duration_sec: Optional[float] = None,
    segment_seconds: Optional[float] = None,
) -> np.ndarray:
    if start_sec < 0:
        raise ValueError("start_sec must be >= 0")
    if duration_sec is not None and duration_sec <= 0:
        raise ValueError("duration_sec must be > 0 when provided")
    if segment_seconds is not None and segment_seconds <= 0:
        raise ValueError("segment_seconds must be > 0 when provided")

    start_sample = int(round(start_sec * sr))
    if start_sample >= len(waveform):
        raise ValueError(
            f"start_sec ({start_sec}s) is beyond the audio length ({len(waveform) / sr:.2f}s)."
        )

    if duration_sec is None:
        clipped = waveform[start_sample:]
    else:
        end_sample = min(len(waveform), start_sample + int(round(duration_sec * sr)))
        clipped = waveform[start_sample:end_sample]

    if segment_seconds is not None:
        target_len = int(round(segment_seconds * sr))
        if len(clipped) < target_len:
            padded = np.zeros(target_len, dtype=np.float32)
            padded[:len(clipped)] = clipped
            clipped = padded
        else:
            clipped = clipped[:target_len]

    if clipped.size == 0:
        raise ValueError("Selected audio region is empty.")

    return clipped.astype(np.float32, copy=False)


def save_csv(matrix: np.ndarray, csv_path: Path, write_header: bool = False) -> None:
    """
    Save as 229 x number_of_frames, i.e. [mel_band, frame].
    Input matrix is [time_frames, mel_bins].
    """
    csv_matrix = matrix.T
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    if write_header:
        header = ",".join([f"frame_{i}" for i in range(csv_matrix.shape[1])])
        np.savetxt(
            csv_path,
            csv_matrix,
            delimiter=",",
            fmt="%.10f",
            header=header,
            comments="",
        )
    else:
        np.savetxt(csv_path, csv_matrix, delimiter=",", fmt="%.10f")


def generate_logmel_csvs(
    audio_path,
    raw_csv_path=None,
    post_bn_csv_path=None,
    start_sec=0.0,
    duration_sec=None,
    segment_seconds=None,
    device="cpu",
    bn_mode="train",
    write_header=False,
):
    """
    Jupyter-friendly function:
    - loads audio
    - extracts raw log-mel
    - extracts post-BN log-mel
    - saves 2 CSV files only
    """

    audio_path = Path(audio_path)

    if device == "cuda" and torch.cuda.is_available():
        torch_device = torch.device("cuda")
    else:
        if device == "cuda":
            print("CUDA requested but not available; falling back to CPU.")
        torch_device = torch.device("cpu")

    if bn_mode not in {"train", "eval"}:
        raise ValueError("bn_mode must be 'train' or 'eval'")

    waveform, orig_sr = load_audio_mono(audio_path)
    waveform = resample_if_needed(waveform, orig_sr)
    waveform = slice_audio(
        waveform=waveform,
        sr=SAMPLE_RATE,
        start_sec=start_sec,
        duration_sec=duration_sec,
        segment_seconds=segment_seconds,
    )

    frontend = PianoTranscriptionFrontendWithBN().to(torch_device)
    frontend.eval()
    if bn_mode == "train":
        frontend.bn0.train()
    else:
        frontend.bn0.eval()

    waveform_tensor = torch.from_numpy(waveform).to(torch_device)
    raw_logmel, post_bn_logmel = frontend(waveform_tensor)

    raw_logmel = raw_logmel[0].detach().cpu().numpy()          # [time_frames, mel_bins]
    post_bn_logmel = post_bn_logmel[0].detach().cpu().numpy()  # [time_frames, mel_bins]

    if raw_csv_path is None:
        #raw_csv_path = audio_path.with_name(f"{audio_path.stem}_logmel.csv")
        raw_csv_path = logmelSPECTROGRAM
    else:
        raw_csv_path = Path(raw_csv_path)

    if post_bn_csv_path is None:
        #post_bn_csv_path = audio_path.with_name(f"{audio_path.stem}_logmel_post_bn.csv")
        post_bn_csv_path = batchmormalSPECTROGRAM
    else:
        post_bn_csv_path = Path(post_bn_csv_path)

    save_csv(raw_logmel, raw_csv_path, write_header=write_header)
    save_csv(post_bn_logmel, post_bn_csv_path, write_header=write_header)

    #print(f"Saved raw CSV:     {raw_csv_path.resolve()}")
    print(f"Saved raw CSV:     {raw_csv_path}")
    #print(f"Saved post-BN CSV: {post_bn_csv_path.resolve()}")
    print(f"Saved post-BN CSV: {post_bn_csv_path}")
    print(
        f"raw_logmel shape [time_frames, mel_bins]: {raw_logmel.shape}\n"
        f"post_bn shape    [time_frames, mel_bins]: {post_bn_logmel.shape}"
    )
    print(
        "Frontend parameters: "
        f"sr={SAMPLE_RATE}, n_fft={WINDOW_SIZE}, hop={HOP_SIZE}, "
        f"n_mels={MEL_BINS}, fmin={FMIN}, fmax={FMAX}, "
        f"bn_momentum={BN_MOMENTUM}, bn_mode={bn_mode}"
    )

    return raw_csv_path, post_bn_csv_path, raw_logmel, post_bn_logmel

audio_file = originalAUDIO   # change this
raw_csv, post_bn_csv, raw_logmel, post_bn_logmel = generate_logmel_csvs(
    audio_path=audio_file,
    start_sec=0.0,
    duration_sec=None,
    segment_seconds=None,
    device="cpu",      # or "cuda"
    #bn_mode="train",   # or "eval"
    bn_mode="eval",   # or "eval"
    write_header=False
)

In [ ]:
# Displaying spectrograms

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

EXPECTED_BANDS = 229


def load_csv_matrix(csv_path):
    csv_path = Path(csv_path)

    try:
        data = np.genfromtxt(csv_path, delimiter=",", dtype=float)
    except Exception as exc:
        raise RuntimeError(f"Failed to read CSV file: {exc}") from exc

    if data.size == 0:
        raise ValueError("CSV file is empty or contains no numeric data.")

    if np.isscalar(data):
        data = np.array([[float(data)]], dtype=float)
    elif data.ndim == 1:
        data = data.reshape(1, -1)

    if np.isnan(data).all():
        raise ValueError(
            "CSV file does not appear to contain numeric data only "
            "(or only contains NaN values)."
        )

    return data


def ensure_band_major_shape(data, expected_bands=EXPECTED_BANDS):
    rows, cols = data.shape
    if rows == expected_bands:
        return data
    if cols == expected_bands:
        return data.T
    return data


def plot_spectrogram_from_csv(
    csv_file,
    output_png=None,
    frame_min=0,
    frame_max=None,
    value_min=0,
    value_max=2,
    show=True
):
    csv_path = Path(csv_file)

    if not csv_path.is_file():
        raise FileNotFoundError(f"File not found: {csv_path}")

    data = load_csv_matrix(csv_path)
    data = ensure_band_major_shape(data, expected_bands=EXPECTED_BANDS)

    rows, cols = data.shape

    if frame_max is None:
        frame_max = cols

    if frame_max <= frame_min:
        raise ValueError("frame_max must be greater than frame_min.")

    if value_max <= value_min:
        raise ValueError("value_max must be greater than value_min.")

    if output_png is None:
        output_png = csv_path.with_name(f"{csv_path.stem}_spectrogram.png")
    else:
        output_png = Path(output_png)

    print(f"File: {csv_path}")
    print(f"Shape: {rows} x {cols}")
    print(f"Min value in CSV: {np.nanmin(data)}")
    print(f"Max value in CSV: {np.nanmax(data)}")

    plt.figure(figsize=(14, 6))
    img = plt.imshow(
        data,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        vmin=value_min,
        vmax=value_max,
        extent=[0, cols, 0, rows],
    )

    plt.xlim(frame_min, frame_max)
    plt.xlabel("Frame")
    plt.ylabel("Log-mel band")
    plt.title(
        f"Spectrogram from CSV\n"
        f"{csv_path.name} | frame range [{frame_min}, {frame_max}] | "
        f"value range [{value_min}, {value_max}]"
    )

    cbar = plt.colorbar(img)
    cbar.set_label("Value")

    plt.tight_layout()
    plt.savefig(output_png, dpi=150)

    if show:
        plt.show()
    else:
        plt.close()

    print(f"Saved spectrogram: {output_png}")

    return data, output_png

csv_file1 = logmelSPECTROGRAM   # change this to your CSV path
csv_file2 = batchmormalSPECTROGRAM  # change this to your CSV path

data1, output_png1 = plot_spectrogram_from_csv(
    csv_file=csv_file1,
    frame_min=0,
    value_min=-40,
    value_max=36
)

data2, output_pn2g = plot_spectrogram_from_csv(
    csv_file=csv_file2,
    frame_min=0,
    value_min=-40,
    value_max=36
)


In [ ]:
#INFERENCE STEP
# Generating a new MIDI file from the reference, original AUDIO file

import sys
!{sys.executable}  example.py --audio_path={originalAUDIO} --output_midi_path={generatedMIDI}

In [ ]:
# Plotting MIDI events from the MIDI file
def midi_to_text_and_plot(midi_path, text_path, plot_path, title_text):
    """
    Decodes a MIDI file, writes its events into a text file, and plots:
    1) Velocity vs. Time (ignoring velocity 0) with Y-axis from 0 to 127
    2) Histogram of Velocity Values with 128 bins and X-axis from 0 to 127
    3) MIDI Note Numbers vs. Time mapped to piano keys (1-88)
    4) Histogram of MIDI Note Numbers (mapped to 1-88) with 88 bins
    Args:
        midi_path (str): Path to the input MIDI file.
        text_path (str): Path to save the output text file.
        plot_path (str): Path to save the generated plot as a PNG file.
        title_text (str): Text to be displayed in the upper left corner of the plot.
    """
    import mido
    import matplotlib.pyplot as plt

    midi_file = mido.MidiFile(midi_path)
    ticks_per_beat = midi_file.ticks_per_beat
    tempo = 500000  # Default 120 BPM tempo (500,000 microseconds per beat)

    # Store extracted data
    velocity_values = []
    velocity_times = []
    note_numbers = []
    note_times = []

    with open(text_path, 'w') as text_file:
        text_file.write(f"MIDI File: {midi_path}\n")
        text_file.write(f"Ticks per Beat: {ticks_per_beat}\n\n")

        absolute_ticks = 0  # Tracks cumulative time in ticks

        for i, track in enumerate(midi_file.tracks):
            text_file.write(f"Track {i}: {track.name}\n")
            text_file.write("-" * 40 + "\n")

            for msg in track:
                absolute_ticks += msg.time  # Convert relative to absolute time

                # Tempo Change Handling (Optional)
                if msg.type == 'set_tempo':
                    tempo = msg.tempo  # Set new tempo

                # Convert MIDI ticks to seconds
                seconds = mido.tick2second(absolute_ticks, ticks_per_beat, tempo)

                # Note-On Event (Velocity > 0)
                if msg.type == 'note_on' and msg.velocity > 0:
                    velocity_values.append(msg.velocity)  # Store velocity
                    velocity_times.append(seconds)         # Store time for velocity
                    note_numbers.append(msg.note)            # Store MIDI note number
                    note_times.append(seconds)               # Store time for note

                    text_file.write(f"Time {seconds:.3f} sec | NOTE_ON  | "
                                    f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n")

                # Note-Off Event
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    text_file.write(f"Time {seconds:.3f} sec | NOTE_OFF | "
                                    f"Note: {msg.note:3d} | Velocity: 0\n")

            text_file.write("\n")

    # Map MIDI note numbers to piano keys: MIDI 21 -> 1, MIDI 108 -> 88.
    mapped_note_numbers = [n - 20 for n in note_numbers]

    # Generate Plots as Subplots if there is any velocity data
    if velocity_values:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Top Left: Velocity vs. Time (set Y-axis from 0 to 127)
        axes[0, 0].scatter(velocity_times, velocity_values, color='b', alpha=0.7, label="Note Velocity")
        axes[0, 0].set_xlabel("Time (seconds)")
        axes[0, 0].set_ylabel("Velocity (0-127)")
        axes[0, 0].set_title("Velocity vs. Time")
        axes[0, 0].legend()
        axes[0, 0].grid()
        axes[0, 0].set_ylim(0, 127)
        axes[0, 0].set_xlim(left=0)
        axes[0, 0].text(-0.07, 1.1, title_text, transform=axes[0, 0].transAxes, fontsize=12, verticalalignment='top')

        # Bottom Left: MIDI Note Numbers vs. Time (mapped to 1-88)
        axes[1, 0].scatter(note_times, mapped_note_numbers, color='purple', alpha=0.7, label="Piano Key")
        axes[1, 0].set_xlabel("Time (seconds)")
        axes[1, 0].set_ylabel("Piano Key (1-88)")
        axes[1, 0].set_title("MIDI Note Numbers vs. Time")
        axes[1, 0].legend()
        axes[1, 0].grid()
        axes[1, 0].set_xlim(left=0)
        axes[1, 0].set_ylim(1, 88)

        # Top Right: Histogram of Velocity Values (set X-axis from 0 to 127)
        axes[0, 1].hist(velocity_values, bins=128, range=(0, 127), color='g', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel("Velocity")
        axes[0, 1].set_ylabel("Number of Occurrences")
        axes[0, 1].set_title("Histogram of Velocity Values")
        axes[0, 1].grid()
        axes[0, 1].set_xlim(0, 127)

        # Bottom Right: Histogram of MIDI Note Numbers (mapped to 1-88)
        axes[1, 1].hist(mapped_note_numbers, bins=88, range=(1, 88), color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel("Piano Key (1-88)")
        axes[1, 1].set_ylabel("Number of Occurrences")
        axes[1, 1].set_title("Histogram of MIDI Note Numbers")
        axes[1, 1].grid()
        axes[1, 1].set_xlim(1, 88)

        plt.tight_layout()
        plt.savefig(plot_path)  # Save plot as PNG file
        plt.show()

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {generatedTXT}")
midi_to_text_and_plot(generatedMIDI, generatedTempTXT, generatedPNG, filename )

In [ ]:
# Filtering out harmonics from the generated MIDI file

try:
    with open(generatedTempTXT, "r") as infile:
        lines = infile.readlines()
except FileNotFoundError:
    print(f"The file {generatedTempTXT} does not exist in the current directory.")
else:
    filtered_lines = []

    for line in lines:
        parts = line.strip().split()

        # look for "... Note: 33 ..."
        if "Note:" in parts:
            try:
                note_index = parts.index("Note:")
                note_value = int(parts[note_index + 1])
            except (ValueError, IndexError):
                continue

            if note_value == note_nr:
                filtered_lines.append(line.strip())

    with open(generatedTXT, "w") as outfile:
        for line in filtered_lines:
            outfile.write(line + "\n")

    print(f"{len(filtered_lines)} lines have been written to '{generatedTXT}'.")

In [ ]:
# Generating MIDI file again (from the filtered data)

import re
from pathlib import Path
import mido
from mido import Message, MidiFile, MidiTrack, MetaMessage

def text_events_to_midi(
    input_txt,
    output_mid=None,
    ticks_per_beat=480,
    tempo_bpm=120
):
    """
    Convert a text MIDI-event log into a .mid file.

    Expected input line format:
    Time 1.003 sec | NOTE_ON  | Note:  33 | Velocity:  92
    Time 1.350 sec | NOTE_OFF | Note:  33 | Velocity: 0
    """

    input_txt = Path(input_txt)

    if output_mid is None:
        output_mid = input_txt.with_suffix(".mid")
    else:
        output_mid = Path(output_mid)

    pattern = re.compile(
        r"Time\s+(?P<time>\d+(?:\.\d+)?)\s+sec\s+\|\s+"
        r"(?P<event>NOTE_ON|NOTE_OFF)\s+\|\s+"
        r"Note:\s+(?P<note>\d+)\s+\|\s+"
        r"Velocity:\s+(?P<velocity>\d+)"
    )

    events = []

    with open(input_txt, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            match = pattern.search(line)
            if not match:
                print(f"Skipping unrecognized line {line_no}: {line}")
                continue

            abs_time_sec = float(match.group("time"))
            event_type = match.group("event")
            note = int(match.group("note"))
            velocity = int(match.group("velocity"))

            events.append({
                "time_sec": abs_time_sec,
                "event": event_type,
                "note": note,
                "velocity": velocity,
            })

    if not events:
        raise ValueError(f"No valid MIDI events found in {input_txt}")

    # Sort by absolute time in case the file is not perfectly ordered
    events.sort(key=lambda e: e["time_sec"])

    midi = MidiFile(ticks_per_beat=ticks_per_beat)
    track = MidiTrack()
    midi.tracks.append(track)

    tempo = mido.bpm2tempo(tempo_bpm)

    # Add tempo so second->tick conversion is reproducible
    track.append(MetaMessage("set_tempo", tempo=tempo, time=0))

    prev_time_sec = 0.0

    for ev in events:
        delta_sec = ev["time_sec"] - prev_time_sec
        if delta_sec < 0:
            raise ValueError("Events are not in chronological order.")

        delta_ticks = int(round(
            mido.second2tick(delta_sec, ticks_per_beat=ticks_per_beat, tempo=tempo)
        ))

        if ev["event"] == "NOTE_ON":
            msg_type = "note_on"
        else:
            msg_type = "note_off"

        track.append(
            Message(
                msg_type,
                note=ev["note"],
                velocity=ev["velocity"],
                time=delta_ticks,
                channel=0,
            )
        )

        prev_time_sec = ev["time_sec"]

    midi.save(output_mid)
    print(f"Saved MIDI to: {output_mid}")
    print(f"Events written: {len(events)}")

    return output_mid


postprocessedMIDI = Path(generatedTXT).with_suffix(".mid")

text_events_to_midi(generatedTXT, postprocessedMIDI)

In [ ]:
# Plotting MIDI events from the generated (and filtered) MIDI file

def midi_to_text_and_plot(midi_path, text_path, plot_path, title_text):
    """
    Decodes a MIDI file, writes its events into a text file, and plots:
    1) Velocity vs. Time (ignoring velocity 0) with Y-axis from 0 to 127
    2) Histogram of Velocity Values with 128 bins and X-axis from 0 to 127
    3) MIDI Note Numbers vs. Time mapped to piano keys (1-88)
    4) Histogram of MIDI Note Numbers (mapped to 1-88) with 88 bins
    Args:
        midi_path (str): Path to the input MIDI file.
        text_path (str): Path to save the output text file.
        plot_path (str): Path to save the generated plot as a PNG file.
        title_text (str): Text to be displayed in the upper left corner of the plot.
    """
    import mido
    import matplotlib.pyplot as plt

    midi_file = mido.MidiFile(midi_path)
    ticks_per_beat = midi_file.ticks_per_beat
    tempo = 500000  # Default 120 BPM tempo (500,000 microseconds per beat)

    # Store extracted data
    velocity_values = []
    velocity_times = []
    note_numbers = []
    note_times = []

    with open(text_path, 'w') as text_file:
        text_file.write(f"MIDI File: {midi_path}\n")
        text_file.write(f"Ticks per Beat: {ticks_per_beat}\n\n")

        absolute_ticks = 0  # Tracks cumulative time in ticks

        for i, track in enumerate(midi_file.tracks):
            text_file.write(f"Track {i}: {track.name}\n")
            text_file.write("-" * 40 + "\n")

            for msg in track:
                absolute_ticks += msg.time  # Convert relative to absolute time

                # Tempo Change Handling (Optional)
                if msg.type == 'set_tempo':
                    tempo = msg.tempo  # Set new tempo

                # Convert MIDI ticks to seconds
                seconds = mido.tick2second(absolute_ticks, ticks_per_beat, tempo)

                # Note-On Event (Velocity > 0)
                if msg.type == 'note_on' and msg.velocity > 0:
                    velocity_values.append(msg.velocity)  # Store velocity
                    velocity_times.append(seconds)         # Store time for velocity
                    note_numbers.append(msg.note)            # Store MIDI note number
                    note_times.append(seconds)               # Store time for note

                    text_file.write(f"Time {seconds:.3f} sec | NOTE_ON  | "
                                    f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n")

                # Note-Off Event
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    text_file.write(f"Time {seconds:.3f} sec | NOTE_OFF | "
                                    f"Note: {msg.note:3d} | Velocity: 0\n")

            text_file.write("\n")

    # Map MIDI note numbers to piano keys: MIDI 21 -> 1, MIDI 108 -> 88.
    mapped_note_numbers = [n - 20 for n in note_numbers]

    # Generate Plots as Subplots if there is any velocity data
    if velocity_values:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Top Left: Velocity vs. Time (set Y-axis from 0 to 127)
        axes[0, 0].scatter(velocity_times, velocity_values, color='b', alpha=0.7, label="Note Velocity")
        axes[0, 0].set_xlabel("Time (seconds)")
        axes[0, 0].set_ylabel("Velocity (0-127)")
        axes[0, 0].set_title("Velocity vs. Time")
        axes[0, 0].legend()
        axes[0, 0].grid()
        axes[0, 0].set_ylim(0, 127)
        axes[0, 0].set_xlim(left=0)
        axes[0, 0].text(-0.07, 1.1, title_text, transform=axes[0, 0].transAxes, fontsize=12, verticalalignment='top')

        # Bottom Left: MIDI Note Numbers vs. Time (mapped to 1-88)
        axes[1, 0].scatter(note_times, mapped_note_numbers, color='purple', alpha=0.7, label="Piano Key")
        axes[1, 0].set_xlabel("Time (seconds)")
        axes[1, 0].set_ylabel("Piano Key (1-88)")
        axes[1, 0].set_title("MIDI Note Numbers vs. Time")
        axes[1, 0].legend()
        axes[1, 0].grid()
        axes[1, 0].set_xlim(left=0)
        axes[1, 0].set_ylim(1, 88)

        # Top Right: Histogram of Velocity Values (set X-axis from 0 to 127)
        axes[0, 1].hist(velocity_values, bins=128, range=(0, 127), color='g', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel("Velocity")
        axes[0, 1].set_ylabel("Number of Occurrences")
        axes[0, 1].set_title("Histogram of Velocity Values")
        axes[0, 1].grid()
        axes[0, 1].set_xlim(0, 127)

        # Bottom Right: Histogram of MIDI Note Numbers (mapped to 1-88)
        axes[1, 1].hist(mapped_note_numbers, bins=88, range=(1, 88), color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel("Piano Key (1-88)")
        axes[1, 1].set_ylabel("Number of Occurrences")
        axes[1, 1].set_title("Histogram of MIDI Note Numbers")
        axes[1, 1].grid()
        axes[1, 1].set_xlim(1, 88)

        plt.tight_layout()
        plt.savefig(plot_path)  # Save plot as PNG file
        plt.show()

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {generatedTXT}")
midi_to_text_and_plot(postprocessedMIDI, generatedTempTXT, generatedPNG, filename )